In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
# Datei einlesen
file_path = "04_61243-0002_de.csv"
df = pd.read_csv(file_path, sep=";", encoding="latin1", on_bad_lines="skip")
display(df)

,,,Tabelle: 61243-0002
"Strompreise fÃ¼r Haushalte: Deutschland, Jahre,",NaN,NaN,NaN
"Jahresverbrauchsklassen, Preisbestandteile",NaN,NaN,NaN
__________,NaN,NaN,NaN
"Â© Statistisches Bundesamt (Destatis), 2026",NaN,NaN,NaN
Stand: 27.04.2026 / 13:28:22,NaN,NaN,NaN


In [2]:
file_path = "04_61243-0002_de.csv"

df_raw = pd.read_csv(
    file_path,
    sep=";",
    encoding="utf-8-sig",
    skiprows=6,
    skipfooter=3,
    engine="python",
    header=None
)

df_raw.columns = [
    "Kategorie",
    "Energie_und_Vertrieb",
    "Kennzeichen_1",
    "Netzkosten",
    "Kennzeichen_2",
    "Steuern_Abgaben_Umlagen",
    "Kennzeichen_3"
]

# Рік
df_raw["Jahr"] = np.where(
    df_raw["Kategorie"].astype(str).str.match(r"^\d{4}$"),
    df_raw["Kategorie"],
    np.nan
)

df_raw["Jahr"] = df_raw["Jahr"].ffill()

# Залишаємо тільки реальні категорії споживання
df = df_raw[
    ~df_raw["Kategorie"].astype(str).str.match(r"^\d{4}$") &
    ~df_raw["Kategorie"].isna()
].copy()

# Прибираємо службові колонки з "e"
df = df.drop(columns=["Kennzeichen_1", "Kennzeichen_2", "Kennzeichen_3"])

# Чистимо числа
price_cols = [
    "Energie_und_Vertrieb",
    "Netzkosten",
    "Steuern_Abgaben_Umlagen"
]

for col in price_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

display(df)

,Kategorie,Energie_und_Vertrieb,Netzkosten,Steuern_Abgaben_Umlagen,Jahr
3,unter 1 000 KWh,0.0925,0.1808,0.1837,2019
4,1 000 bis unter 2 500 KWh,0.0683,0.0938,0.1622,2019
5,2 500 bis unter 5 000 KWh,0.0581,0.0740,0.1556,2019
6,5 000 bis unter 15 000 KWh,0.0547,0.0592,0.1504,2019
7,15 000 KWh und mehr,0.0492,0.0444,0.1432,2019
8,Insgesamt,0.0617,0.0804,0.1571,2019
10,unter 1 000 KWh,0.1213,0.1546,0.1849,2020
11,1 000 bis unter 2 500 KWh,0.0742,0.0992,0.1659,2020
12,2 500 bis unter 5 000 KWh,0.0574,0.0864,0.1589,2020
13,5 000 bis unter 15 000 KWh,0.0577,0.0715,0.1545,2020


In [3]:
df["date_from"] = pd.to_datetime(df["Jahr"].astype(str) + "-01-01")

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [5]:
df["kategorie"] = df["kategorie"].str.replace(" ", "")

In [6]:
df["total_price_eur_per_kwh"] = (
    df["energie_und_vertrieb"] +
    df["netzkosten"] +
    df["steuern_abgaben_umlagen"]
).round(4)

In [7]:
df["is_total"] = df["kategorie"].eq("Insgesamt")

In [8]:
display(df)

,kategorie,energie_und_vertrieb,netzkosten,steuern_abgaben_umlagen,jahr,date_from,total_price_eur_per_kwh,is_total
3,unter1000KWh,0.0925,0.1808,0.1837,2019,2019-01-01,0.4570,False
4,1000bisunter2500KWh,0.0683,0.0938,0.1622,2019,2019-01-01,0.3243,False
5,2500bisunter5000KWh,0.0581,0.0740,0.1556,2019,2019-01-01,0.2877,False
6,5000bisunter15000KWh,0.0547,0.0592,0.1504,2019,2019-01-01,0.2643,False
7,15000KWhundmehr,0.0492,0.0444,0.1432,2019,2019-01-01,0.2368,False
8,Insgesamt,0.0617,0.0804,0.1571,2019,2019-01-01,0.2992,True
10,unter1000KWh,0.1213,0.1546,0.1849,2020,2020-01-01,0.4608,False
11,1000bisunter2500KWh,0.0742,0.0992,0.1659,2020,2020-01-01,0.3393,False
12,2500bisunter5000KWh,0.0574,0.0864,0.1589,2020,2020-01-01,0.3027,False
13,5000bisunter15000KWh,0.0577,0.0715,0.1545,2020,2020-01-01,0.2837,False


In [9]:
df = df.rename(columns={
    "kategorie": "verbrauchskategorie",
    "energie_und_vertrieb": "energie_und_vertrieb_eur_kwh",
    "netzkosten": "netzkosten_eur_kwh",
    "steuern_abgaben_umlagen": "steuern_abgaben_umlagen_eur_kwh",
    "jahr": "jahr",
    "date_from": "datum_von",
    "total_price_eur_per_kwh": "strompreis_gesamt_eur_kwh",
    "is_total": "ist_gesamt"
})

In [10]:
df["verbrauchskategorie"] = df["verbrauchskategorie"].replace({
    "unter1000KWh": "Unter 1000 kWh",
    "1000bisunter2500KWh": "1000 bis unter 2500 kWh",
    "2500bisunter5000KWh": "2500 bis unter 5000 kWh",
    "5000bisunter15000KWh": "5000 bis unter 15000 kWh",
    "15000KWhundmehr": "15000 kWh und mehr",
    "Insgesamt": "Insgesamt"
})

In [11]:
df = df[
    [
        "datum_von",
        "jahr",
        "verbrauchskategorie",
        "energie_und_vertrieb_eur_kwh",
        "netzkosten_eur_kwh",
        "steuern_abgaben_umlagen_eur_kwh",
        "strompreis_gesamt_eur_kwh",
        "ist_gesamt"
    ]
]

In [12]:
display(df)

,datum_von,jahr,verbrauchskategorie,energie_und_vertrieb_eur_kwh,netzkosten_eur_kwh,steuern_abgaben_umlagen_eur_kwh,strompreis_gesamt_eur_kwh,ist_gesamt
3,2019-01-01,2019,Unter 1000 kWh,0.0925,0.1808,0.1837,0.4570,False
4,2019-01-01,2019,1000 bis unter 2500 kWh,0.0683,0.0938,0.1622,0.3243,False
5,2019-01-01,2019,2500 bis unter 5000 kWh,0.0581,0.0740,0.1556,0.2877,False
6,2019-01-01,2019,5000 bis unter 15000 kWh,0.0547,0.0592,0.1504,0.2643,False
7,2019-01-01,2019,15000 kWh und mehr,0.0492,0.0444,0.1432,0.2368,False
8,2019-01-01,2019,Insgesamt,0.0617,0.0804,0.1571,0.2992,True
10,2020-01-01,2020,Unter 1000 kWh,0.1213,0.1546,0.1849,0.4608,False
11,2020-01-01,2020,1000 bis unter 2500 kWh,0.0742,0.0992,0.1659,0.3393,False
12,2020-01-01,2020,2500 bis unter 5000 kWh,0.0574,0.0864,0.1589,0.3027,False
13,2020-01-01,2020,5000 bis unter 15000 kWh,0.0577,0.0715,0.1545,0.2837,False


In [13]:
df.to_csv(
    "04_bereinigte_strompreisbestandteile_LV.csv",
    index=False,
    encoding="utf-8-sig"
)